# Notebook 01 — Dataset Overview
**Feature 1.7 — EDA & Data Understanding | HitRadar Pro**

## Mục tiêu
- Kết nối database `hitradar` và liệt kê analytics views.
- Kiểm tra row count, sample records, và data quality metrics.
- Tóm tắt toàn bộ dataset Spotify/HitRadar trước khi EDA chi tiết.
- Ghi nhận các data quality warnings carry-forward từ Feature 1.5.

In [ ]:
import os, psycopg2, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Đọc password từ biến môi trường — không hardcode
DB_PARAMS = dict(
    host='localhost', port=5432, user='postgres',
    password=os.environ.get('PGPASSWORD', ''),
    dbname='hitradar'
)

conn = psycopg2.connect(**DB_PARAMS)
print('Kết nối thành công:', conn.dsn)

## 1. Danh sách Analytics Views

In [ ]:
df_views = pd.read_sql("""
    SELECT table_name AS view_name
    FROM information_schema.views
    WHERE table_schema = 'analytics'
    ORDER BY table_name
""", conn)
print(f'Tổng số views: {len(df_views)}')
df_views

## 2. Row Counts các view chính

In [ ]:
views_to_count = [
    'analytics.vw_tracks_overview',
    'analytics.vw_tracks_by_decade',
    'analytics.vw_audio_trends',
    'analytics.vw_popularity_stats',
    'analytics.vw_top_artists',
    'analytics.vw_genre_trends',
    'analytics.vw_explicit_by_decade',
    'analytics.vw_duration_trends',
    'analytics.vw_data_quality_report',
    'analytics.vw_ml_training_dataset',
]

cur = conn.cursor()
counts = []
for v in views_to_count:
    cur.execute(f'SELECT COUNT(*) FROM {v}')
    counts.append({'view': v.split('.')[1], 'row_count': cur.fetchone()[0]})

df_counts = pd.DataFrame(counts)
df_counts['row_count'] = df_counts['row_count'].apply(lambda x: f'{x:,}')
df_counts

## 3. Tóm tắt Dataset

In [ ]:
summary = pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM analytics.vw_tracks_overview)           AS tong_tracks,
        (SELECT COUNT(*) FROM analytics.vw_top_artists)               AS artists_co_track,
        (SELECT COUNT(DISTINCT genre_id) FROM analytics.vw_genre_trends) AS so_genres,
        (SELECT COUNT(DISTINCT release_year) FROM analytics.vw_audio_trends) AS so_nam,
        (SELECT COUNT(DISTINCT decade) FROM analytics.vw_tracks_by_decade)   AS so_thap_ky
""", conn)

print('=== TÓM TẮT DATASET ===')
for col in summary.columns:
    print(f'  {col:25s}: {summary[col].values[0]:,}')

## 4. Sample Records

In [ ]:
print('--- vw_tracks_overview (5 records) ---')
df_sample = pd.read_sql("""
    SELECT track_id, name, target_popularity AS popularity, duration_min,
           release_year, decade, release_precision, danceability, energy, valence
    FROM analytics.vw_ml_training_dataset LIMIT 5
""", conn)
df_sample

In [ ]:
print('--- vw_data_quality_report ---')
pd.read_sql('SELECT * FROM analytics.vw_data_quality_report ORDER BY severity DESC, metric_name', conn)

## 5. Biểu đồ phân bổ theo decade

In [ ]:
df_decade = pd.read_sql("""
    SELECT decade, track_count
    FROM analytics.vw_tracks_by_decade
    WHERE decade >= 1920
    ORDER BY decade
""", conn)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(df_decade['decade'].astype(str), df_decade['track_count'],
              color='steelblue', edgecolor='white')
ax.set_title('Số lượng tracks theo thập kỷ', fontsize=14, fontweight='bold')
ax.set_xlabel('Thập kỷ')
ax.set_ylabel('Số tracks')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(f'Thập kỷ nhiều tracks nhất: {df_decade.loc[df_decade.track_count.idxmax(), "decade"]}s '
      f'({df_decade.track_count.max():,} tracks)')

## 6. Data Quality Warnings Carry-Forward

Các cảnh báo sau được ghi nhận từ Feature 1.5 và cần theo dõi trong EDA:

| Warning | Giá trị | Ghi chú |
|---------|---------|--------|
| Duration short (< 10s) | **26 tracks** | Giữ lại theo rule F1.4 |
| Duration long (> 60 min) | **83 tracks** | Giữ lại theo rule F1.4 |
| Loudness > 0 dB | **219 tracks** | Hiếm nhưng hợp lệ |
| track_artists coverage | **96.54%** (skipped 26,224) | Skipped = artist_id không có trong artists.csv |
| artist_relations diff | **1** | ON CONFLICT duplicate pair |
| tracks.name NULL | **71** | Retained by rule |
| artists.followers NULL | **11** | Retained by rule |

## 7. Kết luận

- Dataset gồm **586,672 tracks** từ nhiều thời kỳ (1900–2021), tập trung nhất ở thập kỷ 1990s (108,875) và 2010s (105,245).
- **81,776 artists** có ít nhất 1 track; **4,672 genres** được map qua `artist_genres`.
- Data quality: **PASS_WITH_WARNINGS** — không có lỗi cấu trúc, chỉ có warning về duration outliers và loudness.
- Dataset **đủ điều kiện cho EDA chi tiết** và ML handoff ở Feature 1.8.
- ⚠️ **Nhớ:** `target_popularity` là label ML, KHÔNG dùng làm input feature.

In [ ]:
conn.close()
print('Done — Notebook 01 hoàn thành.')